# Stage 1 — seed cases

The minimum-functionality cases of the suite (§3.1, Table 9): two per function (D21), each a
user intent with a persona, structural constraints where the function leaves a trace (D19), and
the ten-point criteria the judge scores against. Factual-QA cases carry a stored reference.

Generated with LLM help and **reviewed by hand**: a case counts as part of the suite once the
author has marked it in `cases/reviewed.json`. The last cell lists what is still unreviewed.
Every case is written against **one starting trip**, the fixture below. The cases name its
stops and its days, so the fixture is part of the test: it is shown first, and each case
declares which of its parts it leans on.

This notebook's outputs are committed: it is the worked example published with the thesis.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # the subtree, so `src` imports

import json, pathlib, textwrap
import pandas as pd
from src.cases import load_cases, by_priority, suite_version, fixture, CASES_DIR

cases = by_priority(load_cases())
print(f"{len(cases)} cases · suite_version {suite_version()[:12]} (covers the cases and the fixture)")
pd.DataFrame([{
    "case_id": c.case_id, "function": c.function, "provenance": c.provenance, "priority": c.run_priority,
    "intent": c.intent or "—", "n": c.repetitions_n, "tools": " → ".join(c.expected_tool_sequence) or "—",
    "constraints": ("checks" if c.declared_constraints.checks != type(c.declared_constraints.checks)() else "")
                   + (" + predicates" if c.declared_constraints.predicates else "") or "none (advisory)",
    "references": len(c.references),
} for c in cases]).set_index("case_id")

8 cases · suite_version 9db8f4080bf7 (covers the cases and the fixture)


,function,provenance,priority,intent,n,tools,constraints,references
case_id,,,,,,,,
advisory_packing_list,advisory,seeded,1,—,8,—,none (advisory),0
advisory_sintra_full_or_half_day,advisory,seeded,1,—,8,—,none (advisory),0
factual_qa_alfama_history,factual_qa,seeded,1,—,8,read_wikipedia,checks,1
factual_qa_lisbon_november_weather,factual_qa,seeded,1,—,8,—,checks,1
scheduling_belem_morning,scheduling,seeded,1,plan,8,search_places → propose_plan,checks,0
scheduling_move_tram,scheduling,seeded,1,reschedule,8,move_slot,checks,0
venue_lookup_seafood_lisbon,venue_lookup,seeded,1,find,8,search_places,checks,0
venue_lookup_sintra_lunch,venue_lookup,seeded,1,find,8,search_places,checks,0


## The fixture trip

Every repetition of every case starts here. The trip is defined as data in
`cases/fixture.json` and rebuilt before each repetition **through the application's own
API** — the same calls a member's browser makes — so the starting state is one the product
produced, not rows the harness inserted. The precondition gate then refuses to start a
repetition unless the trip is exactly this (invariant 1, D03; `src/reset.py`).

It is deliberately small and fixed: two regions the agent's wrong-region guard can be
wrong about, five saved stops, three of them scheduled, and four days of which two are
empty. Cases are written against it — a stop it does not contain cannot be rescheduled,
and a day outside its range cannot be planned.

In [2]:
fx = fixture()
trip = fx["trip"]
print(f'trip:    {trip["name_prefix"]} sim-NN   {trip["start_date"]} to {trip["end_date"]}')
print("         " + textwrap.fill(trip["description"], 88, subsequent_indent=" " * 9))
print()
print("regions:")
for r in fx["regions"]:
    print(f'  {r["name"]:10} {r["region_type"]:8} {r["description"]}')
print()
scheduled = {s["block"]: s for s in fx["slots"]}
print("saved stops (the title is what a case names in fixture_depends_on):")
for b in fx["blocks"]:
    slot = scheduled.get(b["title"])
    when = f'{slot["day_date"]} {slot["start_time"]}-{slot["end_time"]}' if slot else "unscheduled"
    print(f'  {b["type"]:12} {b["title"]:32} {when}')
print()
days = pd.date_range(trip["start_date"], trip["end_date"]).strftime("%Y-%m-%d")
print("days:")
for day in days:
    on_day = [b["title"] for b in fx["blocks"] if scheduled.get(b["title"], {}).get("day_date") == day]
    print(f'  {day}  {", ".join(on_day) if on_day else "(empty)"}')

trip:    [fixture] sim-NN   2026-11-06 to 2026-11-09
         A four-day Lisbon and Sintra trip the regression suite runs against. Rebuilt before
         every repetition; never edited by hand.

regions:
  Lisbon     city     Base for three days; old town, riverfront, Belém.
  Sintra     area     Day trip: palaces and the hills.

saved stops (the title is what a case names in fixture_depends_on):
  hotel        Hotel near Baixa                 unscheduled
  sightseeing  Tram 28 ride through Alfama      2026-11-06 10:00-11:00
  sightseeing  Jerónimos Monastery              unscheduled
  restaurant   Time Out Market lunch            2026-11-06 12:30-14:00
  sightseeing  Pena Palace                      2026-11-08 10:00-13:00

days:
  2026-11-06  Tram 28 ride through Alfama, Time Out Market lunch
  2026-11-07  (empty)
  2026-11-08  Pena Palace
  2026-11-09  (empty)


## Each case in full

The persona and goal drive the simulator; the opening turn is sent verbatim; the constraints are what the structural checker evaluates; the criteria are what the judge scores.

In [3]:
for c in cases:
    print("═" * 100)
    print(f"{c.case_id}   [{c.function} · {c.provenance} · priority {c.run_priority} · n={c.repetitions_n}]")
    print("persona:  " + textwrap.fill(c.user_profile.strip(), 90, subsequent_indent=" " * 10))
    print("goal:     " + textwrap.fill(c.goal.strip(), 90, subsequent_indent=" " * 10))
    print(f"opens:    {c.opening_turn!r}" + (f"   (intent {c.intent})" if c.intent else ""))
    # What the case requires of the tools, which is not always an order: a
    # question three tools answer legitimately constrains the set, not the
    # sequence. Printing an empty sequence as "no tools" said the opposite.
    if c.expected_tool_sequence:
        expects = " → ".join(c.expected_tool_sequence)
    elif c.declared_constraints.checks.tools_any_of:
        expects = "any one of: " + ", ".join(c.declared_constraints.checks.tools_any_of)
    elif c.declared_constraints.is_empty():
        expects = "not constrained — advisory leaves no structural trace (P4)"
    else:
        expects = "no tool constraint"
    print("expects:  " + expects)
    cons = c.declared_constraints
    if cons.is_empty():
        print("checks:   none — advisory leaves no structural trace (P4)")
    else:
        for k, v in cons.checks.model_dump(exclude_none=True).items():
            print(f"check:    {k} = {json.dumps(v)}")
        if cons.predicates:
            print("predicates: " + ", ".join(cons.predicates))
    for i, crit in enumerate(c.scoring_criteria, 1):
        print(f"criterion {i}: {crit}")
    for ref in c.references:
        print(f"reference: {ref.title} § {ref.section} → references/{ref.path} ({len(ref.text())} chars)")
    print("fixture:  " + (", ".join(c.fixture_depends_on) if c.fixture_depends_on else "nothing — the case reads nothing off the trip"))
    if c.notes:
        print("notes:    " + textwrap.fill(c.notes.strip(), 90, subsequent_indent=" " * 10))

════════════════════════════════════════════════════════════════════════════════════════════════════
advisory_packing_list   [advisory · seeded · priority 1 · n=8]
persona:  Marta, 34, packing this week, wants a checklist she can act on.
goal:     A sensible packing list for four days in Lisbon and Sintra in November.
opens:    '@Tonic what should the three of us pack for this trip?'
expects:  not constrained — advisory leaves no structural trace (P4)
checks:   none — advisory leaves no structural trace (P4)
criterion 1: Fits the trip on the itinerary: November, city walking, a Sintra hill day.
criterion 2: Concrete and short: items, not paragraphs.
criterion 3: Does not claim to have changed the trip.
fixture:  Pena Palace, Tram 28 ride through Alfama, Time Out Market lunch
notes:    Advisory leaves no structural trace (P4), so there is nothing to assert and the judge
          carries the whole case. The fixture dependency is real even so: 'fits the trip
          on the itinerary' i

## Review status

A case is adopted when it appears in `cases/reviewed.json` with who reviewed it and when. Anything listed below is a draft the suite must not run on.

In [4]:
ledger = json.loads((CASES_DIR / "reviewed.json").read_text(encoding="utf-8"))["reviewed"]
unreviewed = [c.case_id for c in cases if c.case_id not in ledger]
for c in cases:
    mark = ledger.get(c.case_id)
    status = mark["by"] + " " + mark["on"] if mark else "unreviewed"
    print(("[x] " if mark else "[ ] ") + f"{c.case_id:40} {status}")
print()
print(f"{len(cases) - len(unreviewed)} of {len(cases)} reviewed" + (" -- the suite is not ready to run" if unreviewed else " -- all adopted"))


[ ] advisory_packing_list                    unreviewed
[ ] advisory_sintra_full_or_half_day         unreviewed
[ ] factual_qa_alfama_history                unreviewed
[ ] factual_qa_lisbon_november_weather       unreviewed
[ ] scheduling_belem_morning                 unreviewed
[ ] scheduling_move_tram                     unreviewed
[ ] venue_lookup_seafood_lisbon              unreviewed
[ ] venue_lookup_sintra_lunch                unreviewed

0 of 8 reviewed -- the suite is not ready to run
